In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# بارگذاری متغیرهای محیطی
load_dotenv()

# مقداردهی مدل جمینای با دمای صفر برای کارهای استدلالی
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0
)

print("✅ Gemini model initialized with temperature=0!")

✅ Gemini model initialized with temperature=0!


In [2]:
# تعریف قالب پرامپت با عبارت کلیدی "مرحله به مرحله فکر کن"
template = """
مسئله زیر را حل کن:
{question}

لطفاً مرحله به مرحله فکر کن و تمام محاسبات را بنویس.
"""

prompt = ChatPromptTemplate.from_template(template)

# ساخت زنجیره با LCEL
chain_zero_shot = prompt | model | StrOutputParser()

print("✅ Zero-Shot CoT Chain is ready!")

✅ Zero-Shot CoT Chain is ready!


In [3]:
question = '''
من ۳ جعبه دارم.
در جعبه اول ۵ سیب قرمز است. در جعبه دوم دو برابر جعبه اول سیب زرد است.
جعبه سوم خالی است.
اگر نصف سیب‌های زرد را به جعبه سوم بریزم، در مجموع چند سیب در جعبه دوم باقی می‌ماند؟
'''

print("⏳ Thinking step by step...\n")

# دریافت پاسخ به صورت قطعه قطعه (Streaming)
for chunk in chain_zero_shot.stream({"question": question}):
    print(chunk, end="", flush=True)

⏳ Thinking step by step...



Direct use of automatic function calling (AFC) in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream. Similarly, direct use of AFC in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message.


برای حل این مسئله، بیایید مرحله به مرحله پیش برویم:

**مرحله ۱: بررسی تعداد سیب‌های جعبه اول**
* در جعبه اول، ۵ سیب قرمز وجود دارد.

**مرحله ۲: محاسبه تعداد سیب‌های جعبه دوم**
* صورت مسئله می‌گوید در جعبه دوم "دو برابر جعبه اول سیب زرد است".
* تعداد سیب‌های جعبه دوم = $۲ \times ۵ = ۱۰$ سیب زرد.

**مرحله ۳: بررسی جعبه سوم**
* جعبه سوم در ابتدا خالی است (۰ سیب).

**مرحله ۴: انتقال سیب‌ها به جعبه سوم**
* صورت مسئله می‌گوید: "اگر نصف سیب‌های زرد را به جعبه سوم بریزم..."
* تعداد کل سیب‌های زرد ۱۰ عدد بود.
* نصف سیب‌های زرد می‌شود: $۱۰ \div ۲ = ۵$ سیب.
* این ۵ سیب از جعبه دوم برداشته شده و به جعبه سوم منتقل می‌شوند.

**مرحله ۵: محاسبه تعداد باقیمانده در جعبه دوم**
* تعداد اولیه سیب‌های جعبه دوم: ۱۰ عدد
* تعداد سیب‌های برداشته شده: ۵ عدد
* تعداد سیب‌های باقیمانده در جعبه دوم = $۱۰ - ۵ = ۵$ عدد.

**پاسخ نهایی:**
در مجموع **۵** سیب در جعبه دوم باقی می‌ماند.

In [4]:
# 1. تعریف متن سیستم (شامل قوانین و مثال‌های استدلال)
system_context = """
تو یک کارشناس ارشد پشتیبانی هستی. وظیفه تو بررسی درخواست مرجوعی است.
برای پاسخ دادن، باید طبق الگوی زیر "استدلال" کنی و سپس "نتیجه نهایی" را بگویی.

قوانین:
1. مرجوعی فقط تا ۳۰ روز مجاز است.
2. کالای الکترونیکی باز شده مرجوع نمی‌شود (مگر نقص فنی).

مثال ۱:
درخواست: "من ۲۵ روز پیش یک لپتاپ خریدم. جعبه را باز کردم ولی رنگش را دوست ندارم."
استدلال:
- گام ۱: بررسی زمان خرید. $25 < 30$ روز است. (شرط زمان: پاس شد)
- گام ۲: بررسی نوع کالا. لپتاپ کالای الکترونیکی است.
- گام ۳: بررسی وضعیت جعبه. کاربر گفته جعبه باز شده است.
- گام ۴: بررسی علت. علت "سلیقه‌ای" است و "نقص فنی" نیست.
- نتیجه‌گیری: طبق قانون ۲، کالای الکترونیکی باز شده پس گرفته نمی‌شود.
نتیجه نهایی: رد درخواست.

مثال ۲:
درخواست: "۴۰ روز پیش یک پیراهن خریدم."
استدلال:
- گام ۱: بررسی زمان خرید. $40 > 30$ روز است. (شرط زمان: رد شد)
- نتیجه‌گیری: طبق قانون ۱، زمان گذشته است.
نتیجه نهایی: رد درخواست.
"""

# 2. ساخت قالب چت
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", system_context),
    ("human", "درخواست جدید: {request}\n\nاستدلال:")
])

# 3. ساخت زنجیره LCEL
chain_few_shot = few_shot_prompt | model | StrOutputParser()

print("✅ Few-Shot CoT Chain is ready!")

✅ Few-Shot CoT Chain is ready!


In [5]:
# 4. تست با یک سناریوی جدید و چالشی (استثنای قانون ۲)
new_request = "من ۱۵ روز پیش یک هدفون خریدم. جعبه را باز کردم و دیدم یکی از گوشی‌ها کار نمی‌کند."

print("⏳ Analyzing return request...\n")
response = chain_few_shot.invoke({"request": new_request})

print(response)

⏳ Analyzing return request...

- گام ۱: بررسی زمان خرید. $15 < 30$ روز است. (شرط زمان: پاس شد)
- گام ۲: بررسی نوع کالا. هدفون یک کالای الکترونیکی است.
- گام ۳: بررسی وضعیت جعبه. کاربر گفته جعبه باز شده است.
- گام ۴: بررسی علت. کاربر گزارش داده که "یکی از گوشی‌ها کار نمی‌کند" که نشان‌دهنده "نقص فنی" است.
- نتیجه‌گیری: طبق استثنای قانون ۲، کالای الکترونیکی باز شده در صورت داشتن نقص فنی قابل مرجوع است.
نتیجه نهایی: تایید درخواست.
